# sklearn-api Database Analysis

This notebook demonstrates how to connect to the PostgreSQL database and run various analytical queries.

## Setup Requirements

Install required packages:
```bash
pip install psycopg2-binary python-dotenv pandas matplotlib seaborn
```

In [1]:
import os
import psycopg2
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime, timedelta
import json

# Load environment variables from .env file
load_dotenv()

print("✅ Imports successful")

✅ Imports successful


## Database Connection

Connect to PostgreSQL using environment variables from `.env` file.

In [2]:
# Get database credentials from environment variables
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5434')
DB_NAME = os.getenv('DB_NAME', 'sklearn_api')
DB_USER = os.getenv('DB_USER', 'postgres')
DB_PASSWORD = os.getenv('DB_PASSWORD')

# Create connection
def get_connection():
    """Create and return a database connection."""
    return psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

# Test connection
try:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT version();')
    db_version = cursor.fetchone()
    print(f"✅ Connected to database: {DB_NAME}")
    print(f"📊 PostgreSQL version: {db_version[0][:50]}...")
    cursor.close()
    conn.close()
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Connected to database: sklearn_api
📊 PostgreSQL version: PostgreSQL 16.11 on x86_64-pc-linux-musl, compiled...


## Helper Function

Utility function to execute queries and return results as pandas DataFrame.

In [3]:
def query_to_dataframe(query, params=None):
    """Execute SQL query and return results as pandas DataFrame."""
    conn = get_connection()
    try:
        df = pd.read_sql_query(query, conn, params=params)
        return df
    finally:
        conn.close()

print("✅ Helper function ready")

✅ Helper function ready


## 1. Database Schema Overview

In [4]:
# List all tables
tables_query = """
SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'public'
ORDER BY table_name;
"""

tables_df = query_to_dataframe(tables_query)
print("📋 Tables in database:")
display(tables_df)

📋 Tables in database:


C:\Users\alan_\AppData\Local\Temp\ipykernel_15388\2668962647.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


,table_name
0,model_metadata
1,predictions
2,users


In [5]:
# Get row counts for each table
row_counts = {}
for table in tables_df['table_name']:
    count_query = f"SELECT COUNT(*) as count FROM {table};"
    result = query_to_dataframe(count_query)
    row_counts[table] = result['count'][0]

counts_df = pd.DataFrame(list(row_counts.items()), columns=['Table', 'Row Count'])
print("📊 Row counts:")
display(counts_df)

📊 Row counts:


C:\Users\alan_\AppData\Local\Temp\ipykernel_15388\2668962647.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


,Table,Row Count
0,model_metadata,3
1,predictions,1
2,users,1


## 2. Users Table Analysis

In [6]:
# View all users
users_query = """
SELECT 
    id,
    username,
    email,
    role,
    is_active,
    rate_limit_per_day,
    created_at,
    last_login
FROM users
ORDER BY created_at;
"""

users_df = query_to_dataframe(users_query)
print(f"👥 Total users: {len(users_df)}")
display(users_df)

👥 Total users: 1


C:\Users\alan_\AppData\Local\Temp\ipykernel_15388\2668962647.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


,id,username,email,role,is_active,rate_limit_per_day,created_at,last_login
0,1,admin,admin@sklearn-api.local,ADMIN,True,10000,2026-04-04 14:24:17.015999,2026-04-04 14:25:25.094054


In [7]:
# User statistics by role
role_stats_query = """
SELECT 
    role,
    COUNT(*) as user_count,
    COUNT(CASE WHEN is_active = true THEN 1 END) as active_count,
    AVG(rate_limit_per_day) as avg_rate_limit
FROM users
GROUP BY role
ORDER BY user_count DESC;
"""

role_stats_df = query_to_dataframe(role_stats_query)
print("📊 Users by role:")
display(role_stats_df)

📊 Users by role:


C:\Users\alan_\AppData\Local\Temp\ipykernel_15388\2668962647.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


,role,user_count,active_count,avg_rate_limit
0,ADMIN,1,1,10000.0


## 3. Model Metadata Analysis

In [8]:
# View all models
models_query = """
SELECT 
    name,
    version,
    is_active,
    is_production,
    algorithm,
    accuracy,
    precision,
    recall,
    f1_score,
    features_count,
    created_at,
    minio_object_key
FROM model_metadata
ORDER BY name, version;
"""

models_df = query_to_dataframe(models_query)
print(f"🤖 Total models: {len(models_df)}")
display(models_df)

🤖 Total models: 3


C:\Users\alan_\AppData\Local\Temp\ipykernel_15388\2668962647.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


,name,version,is_active,is_production,algorithm,accuracy,precision,recall,f1_score,features_count,created_at,minio_object_key
0,cancer_model,1.0.0,True,True,None,None,None,None,None,None,2026-04-04 14:24:57.539831,cancer_model/1.0.0.pkl
1,iris_model,1.0.0,True,True,None,None,None,None,None,None,2026-04-04 14:24:57.534054,iris_model/1.0.0.pkl
2,wine_model,1.0.0,True,True,None,None,None,None,None,None,2026-04-04 14:24:57.478456,wine_model/1.0.0.pkl


In [9]:
# Model status summary
model_status_query = """
SELECT 
    COUNT(*) as total_models,
    COUNT(CASE WHEN is_active = true THEN 1 END) as active_models,
    COUNT(CASE WHEN is_production = true THEN 1 END) as production_models,
    COUNT(DISTINCT name) as unique_model_names
FROM model_metadata;
"""

model_status_df = query_to_dataframe(model_status_query)
print("📊 Model status summary:")
display(model_status_df)

📊 Model status summary:


C:\Users\alan_\AppData\Local\Temp\ipykernel_15388\2668962647.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


,total_models,active_models,production_models,unique_model_names
0,3,3,3,3


## 4. Predictions Analysis

In [10]:
# Recent predictions
recent_predictions_query = """
SELECT 
    p.id,
    u.username,
    p.model_name,
    p.model_version,
    p.status,
    p.response_time_ms,
    p.timestamp
FROM predictions p
LEFT JOIN users u ON p.user_id = u.id
ORDER BY p.timestamp DESC
LIMIT 20;
"""

recent_predictions_df = query_to_dataframe(recent_predictions_query)
print(f"🔮 Recent predictions (last 20):")
display(recent_predictions_df)

🔮 Recent predictions (last 20):


C:\Users\alan_\AppData\Local\Temp\ipykernel_15388\2668962647.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


,id,username,model_name,model_version,status,response_time_ms,timestamp
0,1,admin,iris_model,1.0.0,success,778.134108,2026-04-04 14:25:25.876564


In [11]:
# Prediction statistics by model
model_stats_query = """
SELECT 
    model_name,
    COUNT(*) as total_predictions,
    COUNT(CASE WHEN status = 'success' THEN 1 END) as successful,
    COUNT(CASE WHEN status = 'error' THEN 1 END) as errors,
    ROUND(AVG(response_time_ms)::numeric, 2) as avg_response_time_ms,
    ROUND(MIN(response_time_ms)::numeric, 2) as min_response_time_ms,
    ROUND(MAX(response_time_ms)::numeric, 2) as max_response_time_ms
FROM predictions
GROUP BY model_name
ORDER BY total_predictions DESC;
"""

model_stats_df = query_to_dataframe(model_stats_query)
print("📊 Statistics by model:")
display(model_stats_df)

📊 Statistics by model:


C:\Users\alan_\AppData\Local\Temp\ipykernel_15388\2668962647.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


,model_name,total_predictions,successful,errors,avg_response_time_ms,min_response_time_ms,max_response_time_ms
0,iris_model,1,1,0,778.13,778.13,778.13


In [12]:
# User activity (predictions per user)
user_activity_query = """
SELECT 
    u.username,
    u.role,
    COUNT(p.id) as total_predictions,
    COUNT(CASE WHEN DATE(p.timestamp) = CURRENT_DATE THEN 1 END) as predictions_today,
    u.rate_limit_per_day,
    MAX(p.timestamp) as last_prediction
FROM users u
LEFT JOIN predictions p ON u.id = p.user_id
GROUP BY u.id, u.username, u.role, u.rate_limit_per_day
ORDER BY total_predictions DESC;
"""

user_activity_df = query_to_dataframe(user_activity_query)
print("👤 User activity:")
display(user_activity_df)

👤 User activity:


C:\Users\alan_\AppData\Local\Temp\ipykernel_15388\2668962647.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


,username,role,total_predictions,predictions_today,rate_limit_per_day,last_prediction
0,admin,ADMIN,1,1,10000,2026-04-04 14:25:25.876564


## 5. Time-Based Analysis

In [13]:
# Predictions by date
daily_predictions_query = """
SELECT 
    DATE(timestamp) as date,
    COUNT(*) as predictions,
    COUNT(DISTINCT user_id) as unique_users,
    COUNT(DISTINCT model_name) as models_used,
    ROUND(AVG(response_time_ms)::numeric, 2) as avg_response_time
FROM predictions
GROUP BY DATE(timestamp)
ORDER BY date DESC
LIMIT 30;
"""

daily_predictions_df = query_to_dataframe(daily_predictions_query)
print("📅 Daily prediction activity:")
display(daily_predictions_df)

📅 Daily prediction activity:


C:\Users\alan_\AppData\Local\Temp\ipykernel_15388\2668962647.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


,date,predictions,unique_users,models_used,avg_response_time
0,2026-04-04,1,1,1,778.13


In [14]:
# Predictions by hour of day
hourly_pattern_query = """
SELECT 
    EXTRACT(HOUR FROM timestamp) as hour,
    COUNT(*) as prediction_count,
    ROUND(AVG(response_time_ms)::numeric, 2) as avg_response_time
FROM predictions
GROUP BY EXTRACT(HOUR FROM timestamp)
ORDER BY hour;
"""

hourly_pattern_df = query_to_dataframe(hourly_pattern_query)
print("⏰ Predictions by hour of day:")
display(hourly_pattern_df)

⏰ Predictions by hour of day:


C:\Users\alan_\AppData\Local\Temp\ipykernel_15388\2668962647.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


,hour,prediction_count,avg_response_time
0,14.0,1,778.13


## 6. Error Analysis

In [15]:
# Recent errors
errors_query = """
SELECT 
    p.timestamp,
    u.username,
    p.model_name,
    p.error_message,
    p.client_ip
FROM predictions p
LEFT JOIN users u ON p.user_id = u.id
WHERE p.status = 'error'
ORDER BY p.timestamp DESC
LIMIT 20;
"""

errors_df = query_to_dataframe(errors_query)
print(f"❌ Recent errors: {len(errors_df)}")
if len(errors_df) > 0:
    display(errors_df)
else:
    print("✅ No errors found!")

❌ Recent errors: 0
✅ No errors found!


C:\Users\alan_\AppData\Local\Temp\ipykernel_15388\2668962647.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


## 7. Performance Metrics

In [16]:
# Overall system performance
performance_query = """
SELECT 
    COUNT(*) as total_predictions,
    COUNT(DISTINCT user_id) as unique_users,
    COUNT(DISTINCT model_name) as models_used,
    ROUND(AVG(response_time_ms)::numeric, 2) as avg_response_time,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY response_time_ms)::numeric, 2) as median_response_time,
    ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY response_time_ms)::numeric, 2) as p95_response_time,
    ROUND((COUNT(CASE WHEN status = 'success' THEN 1 END)::float / COUNT(*) * 100)::numeric, 2) as success_rate_percent
FROM predictions;
"""

performance_df = query_to_dataframe(performance_query)
print("⚡ Overall system performance:")
display(performance_df)

⚡ Overall system performance:


C:\Users\alan_\AppData\Local\Temp\ipykernel_15388\2668962647.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


,total_predictions,unique_users,models_used,avg_response_time,median_response_time,p95_response_time,success_rate_percent
0,1,1,1,778.13,778.13,778.13,100.0


## 8. Advanced Queries

In [17]:
# Model comparison matrix
model_comparison_query = """
SELECT 
    model_name,
    COUNT(*) as usage_count,
    COUNT(DISTINCT user_id) as users,
    ROUND((COUNT(CASE WHEN status = 'success' THEN 1 END)::float / COUNT(*) * 100)::numeric, 2) as success_rate,
    ROUND(AVG(response_time_ms)::numeric, 2) as avg_time_ms,
    MIN(timestamp) as first_used,
    MAX(timestamp) as last_used
FROM predictions
GROUP BY model_name
ORDER BY usage_count DESC;
"""

model_comparison_df = query_to_dataframe(model_comparison_query)
print("🔄 Model comparison:")
display(model_comparison_df)

🔄 Model comparison:


C:\Users\alan_\AppData\Local\Temp\ipykernel_15388\2668962647.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


,model_name,usage_count,users,success_rate,avg_time_ms,first_used,last_used
0,iris_model,1,1,100.0,778.13,2026-04-04 14:25:25.876564,2026-04-04 14:25:25.876564


In [18]:
# User engagement scores
engagement_query = """
SELECT 
    u.username,
    COUNT(p.id) as total_predictions,
    COUNT(DISTINCT DATE(p.timestamp)) as active_days,
    COUNT(DISTINCT p.model_name) as models_tried,
    ROUND((COUNT(p.id)::float / GREATEST(COUNT(DISTINCT DATE(p.timestamp)), 1))::numeric, 2) as predictions_per_day,
    MAX(p.timestamp) as last_active
FROM users u
LEFT JOIN predictions p ON u.id = p.user_id
WHERE u.is_active = true
GROUP BY u.id, u.username
ORDER BY total_predictions DESC;
"""

engagement_df = query_to_dataframe(engagement_query)
print("📈 User engagement:")
display(engagement_df)

📈 User engagement:


C:\Users\alan_\AppData\Local\Temp\ipykernel_15388\2668962647.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


,username,total_predictions,active_days,models_tried,predictions_per_day,last_active
0,admin,1,1,1,1.0,2026-04-04 14:25:25.876564


## 9. Custom Query Section

Use this section to run your own custom queries.

In [19]:
# Write your custom query here
custom_query = """
-- Example: Your custom query
SELECT * FROM users LIMIT 5;
"""

custom_df = query_to_dataframe(custom_query)
display(custom_df)

C:\Users\alan_\AppData\Local\Temp\ipykernel_15388\2668962647.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


,id,username,email,api_token,role,is_active,rate_limit_per_day,created_at,updated_at,last_login
0,1,admin,admin@sklearn-api.local,36MBFsfOAKEoHN1Q4m76FQNjryyrZir2p7aBbfbuOz8,ADMIN,True,10000,2026-04-04 14:24:17.015999,2026-04-04 14:25:25.095090,2026-04-04 14:25:25.094054


## Summary

This notebook provides a comprehensive analysis of the sklearn-api database:

1. ✅ Database schema overview
2. 👥 User management and statistics
3. 🤖 Model metadata and versioning
4. 🔮 Prediction logging and analytics
5. 📅 Time-based patterns
6. ❌ Error tracking
7. ⚡ Performance metrics
8. 🔄 Advanced comparative analysis

All queries use environment variables for secure database connection!